# NB_lab_test_trends

Builds `bharath_dev_adb_p1.gold.lab_test_trends` (one row per test_name per calendar month) from `silver.lab_results`, LEFT JOINed to calendar for month/quarter names, with a LAG-based month-over-month abnormal-rate trend. Full rebuild every run per the STTM (Silver to Gold, v1.0).

In [0]:
%sql
CREATE OR REPLACE TABLE bharath_dev_adb_p1.gold.lab_test_trends
COMMENT 'Gold: one row per test_name per month with volume, abnormal/critical rates and month-over-month abnormal-rate trend (STTM Silver to Gold v1.0)'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                            AS year,
    month(test_date)                           AS month,
    COUNT(*)                                   AS total_tests,
    COUNT_IF(result_status = 'Abnormal')       AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')       AS critical_test_count
  FROM bharath_dev_adb_p1.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    m.*,
    CAST(ROUND(abnormal_test_count * 100.0 / total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(critical_test_count * 100.0 / total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  w.test_name,
  CAST(w.year AS INT)                                  AS year,
  CAST(w.month AS INT)                                 AS month,
  c.month_name,
  c.quarter_name,
  CAST(w.total_tests AS INT)                           AS total_tests,
  CAST(w.abnormal_test_count AS INT)                   AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                   AS critical_test_count,
  w.abnormal_rate_pct,
  w.critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2)) AS prev_month_abnormal_rate_pct,
  CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct  -- NULL when no prior month
FROM with_lag w
LEFT JOIN bharath_dev_adb_p1.silver.calendar c
  ON c.date_key = CAST(date_format(make_date(w.year, w.month, 1), 'yyyyMMdd') AS INT);

In [0]:
# Return the gold table record count to the caller (e.g. a job or dbutils.notebook.run)
gold_df = spark.table("bharath_dev_adb_p1.gold.lab_test_trends")

record_count = gold_df.count()
print(f"[STEP 5] Exiting notebook with processed record count: {record_count}")
dbutils.notebook.exit(str(record_count))